# BRAIN Pipeline 分步教学（Notebook）

这个 notebook 是 `run_pipeline.py` 的“教学友好”分步版本：每一步都有明确目的、产出物（文件）、以及检查点，适合课堂/自学逐步运行与讲解。请学会自己debug！

## 总目标（你最终要得到什么）
- 一份可复用的 ideas 报告（Markdown）：用于描述每个 Concept 的直觉、字段、以及实现示例。
- 一份 dataset 的字段清单 CSV：用于验证模板里的占位符确实能匹配真实字段。
- 一堆实现结果文件 `idea_*.json`：每个模板（或每个 Concept）生成的一批表达式候选。
- 合并后的最终结果 `final_expressions.json`：去重后的 expression 列表，可直接用于后续筛选/回测。

## 产出物（课堂最常看的“文件落地”）
- ideas 文件：`skills/brain-data-feature-engineering/output_report/<REGION>_delay<DELAY>_<DATASET_ID>_ideas.md`
- dataset CSV：`skills/brain-feature-implementation/data/<dataset>_<region>_delay<delay>/<dataset>_<region>_delay<delay>.csv`
- 实现输出：`skills/brain-feature-implementation/data/<dataset>_<region>_delay<delay>/idea_<timestamp>.json`
- 合并结果：`skills/brain-feature-implementation/data/<dataset>_<region>_delay<delay>/final_expressions.json`

## 为什么要这样做（教学版关键逻辑）
1) LLM 负责“想法与模板”（ideas.md），但模板必须能落到真实字段；
2) 用 dataset CSV 的 field ids 来校验/规范化模板占位符，避免“看起来对但跑不动”；
3) `implement_idea.py` 做“枚举式实现”：把一个模板扩展成很多个具体表达式；
4) 最后 merge 去重，得到可用的表达式池。

In [ ]:
# ==== 参数区（先改这一格） ====
from pathlib import Path
import os

# 指向 trailSomeAlphas 文件夹（run_pipeline.py 所在目录）
TRAIL_DIR = Path(r"d:\BRAINProject\cnhkmcp\cnhkmcp\untracked\APP\trailSomeAlphas").resolve()

# 核心运行参数
DATA_CATEGORY = "analyst"
REGION = "GLB"
DELAY = 1
UNIVERSE = "TOP3000"
INSTRUMENT_TYPE = "EQUITY"
DATASET_ID = "analyst14"

# ideas Markdown 的处理方式
# - 如果 IDEAS_FILE 已存在且 REGEN_IDEAS=False：直接复用（不调用 LLM）。
# - 否则：会（可选）调用 Moonshot 生成 ideas，并写入 IDEAS_FILE。
IDEAS_FILE = TRAIL_DIR / "skills" / "brain-data-feature-engineering" / "output_report" / f"{REGION}_delay{DELAY}_{DATASET_ID}_ideas.md"
REGEN_IDEAS = False  # 设为 True 会强制重新调用 Moonshot 生成 ideas

# 模板执行策略：有些 ideas 文件会出现“不同 Concept 但模板重复”的情况
# - True  => 同一个 template 只跑一次（更快，冗余更少）
# - False => 每个 Concept 都跑一次（更贴近原报告）
DEDUP_TEMPLATES = True

# Moonshot（仅在需要生成 ideas 时使用）
MOONSHOT_MODEL = "kimi-k2.5"
# 注意：比赛不能展示密钥
MOONSHOT_API_KEY = "sk-xxxxxx"

# Prompt 规模控制
MAX_FIELDS = None      # None => 随机抽样 50 条（或总数 <50 时取全部）
MAX_OPERATORS = 300
NO_OPERATORS_IN_PROMPT = False

print("TRAIL_DIR=", TRAIL_DIR)
print("IDEAS_FILE=", IDEAS_FILE)
print("DEDUP_TEMPLATES=", DEDUP_TEMPLATES)


TRAIL_DIR= D:\BRAINProject\cnhkmcp\cnhkmcp\untracked\APP\trailSomeAlphas
IDEAS_FILE= D:\BRAINProject\cnhkmcp\cnhkmcp\untracked\APP\trailSomeAlphas\skills\brain-data-feature-engineering\output_report\GLB_delay1_analyst14_ideas.md
DEDUP_TEMPLATES= True


In [31]:
# 导入依赖 + 配置路径（与 run_pipeline.py 保持一致）
import json
import re
import csv
import time
import subprocess
import sys

import requests

SKILLS_DIR = TRAIL_DIR / "skills"
FEATURE_ENGINEERING_DIR = SKILLS_DIR / "brain-data-feature-engineering"
FEATURE_IMPLEMENTATION_DIR = SKILLS_DIR / "brain-feature-implementation"
FEATURE_IMPLEMENTATION_SCRIPTS = FEATURE_IMPLEMENTATION_DIR / "scripts"

sys.path.insert(0, str(FEATURE_IMPLEMENTATION_SCRIPTS))
import ace_lib  # type: ignore

print("FEATURE_IMPLEMENTATION_SCRIPTS=", FEATURE_IMPLEMENTATION_SCRIPTS)

FEATURE_IMPLEMENTATION_SCRIPTS= D:\BRAINProject\cnhkmcp\cnhkmcp\untracked\APP\trailSomeAlphas\skills\brain-feature-implementation\scripts


## Step 1) 辅助函数（与 run_pipeline.py 同逻辑）
**目的**：把后续步骤要用的小工具函数提前定义好，避免“每一步都在解释工具细节”。

**本 Step 的产出**：
- 没有文件落地；会在内存里定义一组函数（比如：读取文本、抽取 Concept、规范化占位符、跑脚本等）。

**你应该看到什么**：
- 运行时通常不会有大量输出；后续 Step 会调用这些函数。

In [32]:
def load_brain_credentials(config_path: Path) -> tuple[str, str]:
    if not config_path.exists():
        raise FileNotFoundError(f"Config not found: {config_path}")
    with config_path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    creds = data.get("BRAIN_CREDENTIALS", {})
    email = creds.get("email")
    password = creds.get("password")
    if not email or not password:
        raise ValueError("BRAIN_CREDENTIALS missing in config.json")
    return email, password


def start_brain_session(email: str, password: str):
    ace_lib.get_credentials = lambda: (email, password)
    return ace_lib.start_session()


def pick_first_present_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    lower_map = {col.lower(): col for col in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None


def build_field_summary(fields_df, max_fields: int | None = None, default_sample_size: int = 50):
    id_col = pick_first_present_column(fields_df, ["id", "field_id", "fieldId"])
    desc_col = pick_first_present_column(fields_df, ["description", "desc"])

    if max_fields is None:
        total = int(fields_df.shape[0])
        n = min(default_sample_size, total)
        subset = fields_df if n >= total else fields_df.sample(n=n, random_state=42)
    else:
        total = int(fields_df.shape[0])
        n = min(int(max_fields), total)
        subset = fields_df.head(n)

    rows = []
    for _, row in subset.iterrows():
        rows.append({"id": row.get(id_col), "description": row.get(desc_col)})
    return rows, fields_df.shape[0]


def read_text_optional(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8")
    except Exception:
        return ""


def detect_dataset_code(dataset_ids: list[str]) -> str | None:
    if not dataset_ids:
        return None
    counts: dict[str, int] = {}
    for fid in dataset_ids:
        tok = (str(fid).split("_", 1)[0] or "").strip()
        if tok:
            counts[tok] = counts.get(tok, 0) + 1
    if not counts:
        return None
    return max(counts.items(), key=lambda kv: kv[1])[0]


def ensure_metadata_block(markdown_text: str, dataset_id: str, region: str, delay: int) -> str:
    has_dataset = re.search(r"^\*\*Dataset\*\*:\s*\S+", markdown_text, flags=re.MULTILINE) is not None
    has_region = re.search(r"^\*\*Region\*\*:\s*\S+", markdown_text, flags=re.MULTILINE) is not None
    has_delay = re.search(r"^\*\*Delay\*\*:\s*\d+", markdown_text, flags=re.MULTILINE) is not None
    if has_dataset and has_region and has_delay:
        return markdown_text

    block = ["", f"**Dataset**: {dataset_id}", f"**Region**: {region}", f"**Delay**: {delay}", ""]
    lines = markdown_text.splitlines()
    insert_at = 0
    for i, line in enumerate(lines[:10]):
        if line.strip():
            insert_at = i + 1
            break
    new_lines = lines[:insert_at] + block + lines[insert_at:]
    return "\n".join(new_lines).lstrip("\n")


def load_dataset_ids_from_csv(dataset_csv_path: Path) -> list[str]:
    if not dataset_csv_path.exists():
        return []
    ids: list[str] = []
    with dataset_csv_path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        if "id" not in (reader.fieldnames or []):
            return []
        for row in reader:
            v = (row.get("id") or "").strip()
            if v:
                ids.append(v)
    return ids


def build_allowed_suffixes_from_ids(dataset_ids: list[str], max_suffixes: int = 300) -> list[str]:
    counts: dict[str, int] = {}
    for raw in dataset_ids:
        parts = [p for p in str(raw).split("_") if p]
        if len(parts) < 2:
            continue
        for n in range(1, 6):
            if len(parts) >= n:
                suffix = "_".join(parts[-n:])
                if suffix.replace("_", "").isdigit():
                    continue
                if n == 1 and len(suffix) < 8:
                    continue
                if len(suffix) < 6:
                    continue
                counts[suffix] = counts.get(suffix, 0) + 1

    ranked = sorted(counts.items(), key=lambda kv: (kv[1], kv[0].count("_"), len(kv[0])), reverse=True)
    suffixes: list[str] = []
    for suffix, _ in ranked:
        if suffix not in suffixes:
            suffixes.append(suffix)
        if len(suffixes) >= max_suffixes:
            break
    return suffixes


def compress_to_known_suffix(var: str, allowed_suffixes: list[str]) -> str | None:
    v = var.lower()
    for sfx in sorted(allowed_suffixes, key=len, reverse=True):
        if v.endswith(sfx.lower()):
            return sfx
    return None


def placeholder_is_reasonably_matchable(var: str, dataset_ids: list[str]) -> bool:
    v = var
    if len(v) <= 3:
        pat = re.compile(rf"(^|_){re.escape(v)}(_|$)", flags=re.IGNORECASE)
        return any(pat.search(str(fid)) for fid in dataset_ids)
    return any(v in str(fid) for fid in dataset_ids)


def normalize_template_placeholders(template: str, dataset_ids: list[str], allowed_suffixes: list[str], dataset_code: str | None):
    vars_in_template = re.findall(r"\{([A-Za-z0-9_]+)\}", template)
    if not vars_in_template:
        return template, False

    mapping: dict[str, str] = {}
    for var in set(vars_in_template):
        new_var = var
        if dataset_code and new_var.lower().startswith(dataset_code.lower() + "_"):
            new_var = new_var[len(dataset_code) + 1 :]
        compressed = compress_to_known_suffix(new_var, allowed_suffixes)
        if compressed:
            new_var = compressed
        mapping[var] = new_var

    normalized = template
    for src, dst in mapping.items():
        normalized = normalized.replace("{" + src + "}", "{" + dst + "}")

    vars_after = re.findall(r"\{([A-Za-z0-9_]+)\}", normalized)
    ok = all(placeholder_is_reasonably_matchable(v, dataset_ids) for v in vars_after)
    return normalized, ok


def safe_dataset_id(dataset_id: str) -> str:
    return "".join([c for c in dataset_id if c.isalnum() or c in ("-", "_")])


def run_script(args_list: list[str], cwd: Path):
    result = subprocess.run(args_list, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            "Command failed: " + " ".join(args_list) + f"\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
        )
    return result.stdout


def extract_template_blocks(markdown_text: str) -> list[dict[str, str]]:
    concept_re = re.compile(r"^\*\*Concept\*\*\s*:\s*(.*)\s*$")
    impl_re = re.compile(r"\*\*Implementation Example\*\*\s*:\s*(.*)$", flags=re.IGNORECASE)
    backtick_re = re.compile(r"`([^`]*)`")
    boundary_re = re.compile(r"^(?:-{3,}|#{1,6}\s+.*)\s*$")

    lines = markdown_text.splitlines()
    blocks: list[list[str]] = []
    current: list[str] = []

    def _flush():
        nonlocal current
        if current:
            while current and not current[0].strip():
                current.pop(0)
            while current and not current[-1].strip():
                current.pop()
            if current:
                blocks.append(current)
        current = []

    for line in lines:
        if concept_re.match(line.strip()):
            _flush()
            current = [line]
            continue
        if current and boundary_re.match(line.strip()):
            _flush()
            continue
        if current:
            current.append(line)

    _flush()

    out: list[dict[str, str]] = []
    for block_lines in blocks:
        template: str | None = None
        impl_line_idx: int | None = None

        for i, raw in enumerate(block_lines):
            m = impl_re.search(raw)
            if not m:
                continue
            impl_line_idx = i
            tail = (m.group(1) or "").strip()
            bt = backtick_re.search(tail)
            if bt:
                template = bt.group(1).strip()
                break
            if tail and ("{" in tail and "}" in tail):
                template = tail.strip().strip("`")
                break
            for j in range(i + 1, min(i + 4, len(block_lines))):
                nxt = block_lines[j].strip()
                if not nxt:
                    continue
                bt2 = backtick_re.search(nxt)
                if bt2:
                    template = bt2.group(1).strip()
                    break
                if "{" in nxt and "}" in nxt:
                    template = nxt.strip().strip("`")
                    break
            break

        if not template or "{" not in template or "}" not in template:
            continue

        idea_lines: list[str] = []
        for i, raw in enumerate(block_lines):
            if impl_line_idx is not None and i == impl_line_idx:
                continue
            idea_lines.append(raw)

        idea = "\n".join(idea_lines).strip()
        out.append({"template": template.strip(), "idea": idea})

    return out

## Step 2) 启动 BRAIN 会话
**目的**：登录 BRAIN 平台，拿到可用的 session，后面才能拉 datasets/datafields/operators，以及下载数据。

**输入**：
- `skills/brain-feature-implementation/config.json` 中的账号密码。

**产出**：
- 内存变量 `session`（不是文件）。

**检查点**：
- 代码格会打印类似 `Session started.`；如果失败，优先检查账号/网络/验证码等。

In [33]:
config_path = FEATURE_IMPLEMENTATION_DIR / "config.json"
email, password = load_brain_credentials(config_path)
session = start_brain_session(email, password)
print("Session started.")

Session started.


## Step 3) 读取 / 生成 ideas 报告（Markdown）
**目的**：拿到一份结构化 ideas 报告（Markdown），里面每个 `**Concept**` 区块都应包含：背景描述 + `**Implementation Example**` 模板。

**两种模式**：
- 复用模式（推荐课堂演示）：`IDEAS_FILE` 已存在且 `REGEN_IDEAS=False`，直接读取，不调用 LLM（可重复、稳定）。
- 生成模式：调用 Moonshot/Kimi 生成，并写回 `IDEAS_FILE`（更灵活，但依赖 key/网络/模型输出稳定性）。

**产出物（落地文件）**：
- `skills/brain-data-feature-engineering/output_report/<REGION>_delay<DELAY>_<DATASET_ID>_ideas.md`

**检查点**：
- 输出里会提示“Reusing ideas file”或“Generated ideas file”。
- 最后会打印“Dataset id (from ideas markdown)= ...”，并确保 ideas 头部包含 `Dataset/Region/Delay` 元数据。

In [34]:
# 决定：复用 ideas 文件，还是重新生成 ideas
ideas_path = Path(IDEAS_FILE).resolve() if IDEAS_FILE else None
if ideas_path is None:
    raise ValueError("IDEAS_FILE must be set in this notebook version.")

if ideas_path.exists() and not REGEN_IDEAS:
    ideas_text = ideas_path.read_text(encoding="utf-8")
    print("Reusing ideas file:", ideas_path)
else:
    if not MOONSHOT_API_KEY:
        raise ValueError("MOONSHOT_API_KEY missing. Set env var MOONSHOT_API_KEY or set MOONSHOT_API_KEY in parameters cell.")

    # 为生成 prompt 拉取 datasets + datafields（用于让 LLM 了解字段范围）
    datasets_df = ace_lib.get_datasets(
        session,
        instrument_type=INSTRUMENT_TYPE,
        region=REGION,
        delay=DELAY,
        universe=UNIVERSE,
        theme="ALL",
    )

    dataset_name = None
    dataset_description = None
    id_col = pick_first_present_column(datasets_df, ["id", "dataset_id", "datasetId"])
    name_col = pick_first_present_column(datasets_df, ["name", "dataset_name", "datasetName"])
    desc_col = pick_first_present_column(datasets_df, ["description", "desc", "dataset_description"])
    if id_col:
        matched = datasets_df[datasets_df[id_col].astype(str) == str(DATASET_ID)]
        if not matched.empty:
            row = matched.iloc[0]
            dataset_name = row.get(name_col) if name_col else None
            dataset_description = row.get(desc_col) if desc_col else None

    fields_df = ace_lib.get_datafields(
        session,
        instrument_type=INSTRUMENT_TYPE,
        region=REGION,
        delay=DELAY,
        universe=UNIVERSE,
        dataset_id=DATASET_ID,
        data_type="ALL",
    )

    fields_summary, field_count = build_field_summary(fields_df, max_fields=MAX_FIELDS)
    feature_engineering_skill_md = read_text_optional(FEATURE_ENGINEERING_DIR / "SKILL.md")
    feature_implementation_skill_md = read_text_optional(FEATURE_IMPLEMENTATION_DIR / "SKILL.md")

    # 为了让 notebook 更易读：复用脚本里的 prompt 构建逻辑（run_pipeline.py）
    # 如果导入失败，也可以把 build_prompt/call_moonshot 逻辑直接内联到 notebook
    import run_pipeline as rp

    allowed_metric_suffixes = rp.build_allowed_metric_suffixes(fields_df, max_suffixes=300)

    allowed_operators = []
    if not NO_OPERATORS_IN_PROMPT:
        try:
            operators_df = ace_lib.get_operators(session)
            keep_vector = rp._vector_ratio_from_datafields_df(fields_df) > 0.5
            _, allowed_ops, _ = rp.filter_operators_df(operators_df, keep_vector=keep_vector)
            allowed_operators = allowed_ops[:MAX_OPERATORS] if MAX_OPERATORS else allowed_ops
        except Exception as exc:
            print("Warning: failed to fetch/filter operators; continuing without operators.", exc)

    system_prompt, user_prompt = rp.build_prompt(
        dataset_id=DATASET_ID,
        dataset_name=dataset_name,
        dataset_description=dataset_description,
        data_category=DATA_CATEGORY,
        region=REGION,
        delay=DELAY,
        universe=UNIVERSE,
        fields_summary=fields_summary,
        field_count=field_count,
        feature_engineering_skill_md=feature_engineering_skill_md,
        feature_implementation_skill_md=feature_implementation_skill_md,
        allowed_metric_suffixes=allowed_metric_suffixes,
        allowed_operators=allowed_operators,
    )

    ideas_text = rp.call_moonshot(MOONSHOT_API_KEY, MOONSHOT_MODEL, system_prompt, user_prompt)
    ideas_path.parent.mkdir(parents=True, exist_ok=True)
    ideas_path.write_text(ideas_text, encoding="utf-8")
    print("Generated ideas file:", ideas_path)

# 确保 ideas 头部有 metadata（dataset/region/delay）
ideas_text = ensure_metadata_block(ideas_text, dataset_id=DATASET_ID, region=REGION, delay=DELAY)
ideas_path.write_text(ideas_text, encoding="utf-8")

dataset_id_match = re.search(r"\*\*Dataset\*\*:\s*(\S+)", ideas_text)
dataset_id_from_file = dataset_id_match.group(1) if dataset_id_match else DATASET_ID
print("Dataset id (from ideas markdown)=", dataset_id_from_file)

Reusing ideas file: D:\BRAINProject\cnhkmcp\cnhkmcp\untracked\APP\trailSomeAlphas\skills\brain-data-feature-engineering\output_report\GLB_delay1_analyst14_ideas.md
Dataset id (from ideas markdown)= analyst14


## Step 4) 通过 `fetch_dataset.py` 下载 dataset CSV
**目的**：把目标 dataset 的 field id 清单下载成 CSV（这一步是“模板占位符可落地”的关键依据）。

**输入**：
- dataset id（从 ideas.md 里解析得到的 `dataset_id_from_file`，或回退到参数 `DATASET_ID`）
- `REGION` / `DELAY` / `UNIVERSE` / `INSTRUMENT_TYPE`

**产出物（落地文件）**：
- `skills/brain-feature-implementation/data/<dataset>_<region>_delay<delay>/<dataset>_<region>_delay<delay>.csv`

**检查点**：
- 代码会打印 dataset folder 与 CSV path。
- 还会打印 `Loaded field ids`（字段数量）、`Derived suffix candidates`（用于占位符压缩/匹配）、`Detected dataset_code`（字段前缀统计）。

In [35]:
fetch_script = FEATURE_IMPLEMENTATION_SCRIPTS / "fetch_dataset.py"
run_script(
    [
        sys.executable,
        str(fetch_script),
        "--datasetid",
        dataset_id_from_file,
        "--region",
        REGION,
        "--delay",
        str(DELAY),
        "--universe",
        UNIVERSE,
        "--instrument-type",
        INSTRUMENT_TYPE,
    ],
    cwd=FEATURE_IMPLEMENTATION_SCRIPTS,
)

dataset_folder = f"{safe_dataset_id(dataset_id_from_file)}_{REGION}_delay{DELAY}"
dataset_csv_path = FEATURE_IMPLEMENTATION_DIR / "data" / dataset_folder / f"{dataset_folder}.csv"
print("Dataset folder=", dataset_folder)
print("CSV path=", dataset_csv_path)

dataset_ids = load_dataset_ids_from_csv(dataset_csv_path)
allowed_suffixes = build_allowed_suffixes_from_ids(dataset_ids, max_suffixes=300) if dataset_ids else []
dataset_code = detect_dataset_code(dataset_ids) if dataset_ids else None

print("Loaded field ids:", len(dataset_ids))
print("Derived suffix candidates:", len(allowed_suffixes))
print("Detected dataset_code:", dataset_code)

Dataset folder= analyst14_GLB_delay1
CSV path= D:\BRAINProject\cnhkmcp\cnhkmcp\untracked\APP\trailSomeAlphas\skills\brain-feature-implementation\data\analyst14_GLB_delay1\analyst14_GLB_delay1.csv
Loaded field ids: 925
Derived suffix candidates: 300
Detected dataset_code: anl14


## Step 5) 抽取 Concept 区块 → (template, idea)
**目的**：把 ideas.md 从“人类可读的报告”变成“机器可执行的任务列表”。每个任务是一个二元组：
- `template`：来自 `**Implementation Example**`（优先取反引号里的表达式）
- `idea`：该 Concept 区块里除了 Implementation Example 之外的文字（用于给 implement 提供上下文）

**输入**：
- Step 3 得到的 `ideas_text`
- Step 4 得到的 dataset field ids（用于占位符规范化/校验）

**产出（内存结构）**：
- `runs`：最终要执行的列表 `[(template, idea), ...]`
- 关键统计：Concept 数量、Pairs 数量、Unique templates、Planned runs。

**检查点（课堂讲解重点）**：
- `Pairs after normalization/validation`：说明模板占位符校验通过的数量。
- `Planned runs`：真正会跑多少次 implement。
  - 如果 `DEDUP_TEMPLATES=True`，可能少于 Concept 数量（因为模板重复会被去重）。
  - 如果你想“每个 Concept 都跑一次”，把 `DEDUP_TEMPLATES=False`。

In [36]:
block_pairs = extract_template_blocks(ideas_text)
print("Concept blocks found with templates:", len(block_pairs))

# 规范化 + 校验：确保占位符能匹配真实 dataset field ids
normalized_pairs: list[tuple[str, str]] = []
for item in block_pairs:
    t = str(item.get("template") or "").strip()
    idea_text = str(item.get("idea") or "").strip()
    if not t:
        continue

    if dataset_ids and allowed_suffixes:
        normalized_t, ok = normalize_template_placeholders(t, dataset_ids, allowed_suffixes, dataset_code)
        if not ok:
            continue
        normalized_pairs.append((normalized_t, idea_text))
    else:
        normalized_pairs.append((t, idea_text))

print("Pairs after normalization/validation:", len(normalized_pairs))

# 统计重复模板（同一个 template 可能在多个 Concept 里重复出现）
template_counts: dict[str, int] = {}
for t, _ in normalized_pairs:
    template_counts[t] = template_counts.get(t, 0) + 1

dups = {t: c for t, c in template_counts.items() if c > 1}
print("Unique templates:", len(template_counts))
print("Duplicate templates:", len(dups))
if dups:
    print("\nDuplicates:")
    for t, c in sorted(dups.items(), key=lambda kv: (-kv[1], kv[0])):
        print(f"- {c}x {t}")

# 选择执行计划（是否按模板去重）
if DEDUP_TEMPLATES:
    # 按 template 去重；优先保留第一个非空 idea
    template_to_idea: dict[str, str] = {}
    for t, idea_text in normalized_pairs:
        if t not in template_to_idea or (not template_to_idea[t] and idea_text):
            template_to_idea[t] = idea_text

    runs: list[tuple[str, str]] = [(t, template_to_idea[t]) for t in sorted(template_to_idea.keys())]
else:
    # 不去重：按原顺序逐条执行（包含重复模板）
    runs = normalized_pairs

print("Planned runs:", len(runs))

# 预览前 10 个即将执行的模板
for i, (t, _) in enumerate(runs[:10], start=1):
    print(f"{i:02d}. {t}")


Concept blocks found with templates: 16
Pairs after normalization/validation: 16
Unique templates: 16
Duplicate templates: 0
Planned runs: 16
01. divide(subtract({eps_fp1}, {eps_fp1}), {eps_fp1})
02. divide({ebit_fy2}, {ebit_fp1})
03. divide({ebitda_fp1}, {ebitda_fp1})
04. divide({ntp_fp1}, abs({ntp_fp1}))
05. divide({revenue_fp1}, ts_mean({revenue_fp1}, 252))
06. multiply({ntp_fp1}, inverse({ntp_fp1}))
07. quantile({ebitda_fp1}, driver="gaussian", sigma=1.0)
08. sigmoid(multiply(divide({ntprep_fp1}, {ntprep_fp1}), 0.5))
09. subtract(divide({eps_fy2}, {eps_fy1}), 1)
10. subtract(ts_mean({eps_fp1}, 20), ts_mean({eps_fp1}, 60))


## Step 6) 实现每个模板（调用 `implement_idea.py`）
**目的**：把“一个带占位符的模板”扩展成“一批具体可用的表达式候选”。这是表达式池生成的核心步骤。

**输入**：
- Step 5 的 `runs`（每个元素包含 `template` 与 `idea`）
- Step 4 的 `dataset_folder`（决定输出落地位置）

**产出物（落地文件）**：
- 每跑一次，会在 dataset 目录下生成一个：`idea_<timestamp>.json`
- 这些文件里通常包含：本次 template 的实现表达式列表、以及一些元信息。

**检查点**：
- 控制台会打印 `[idx/total] Implementing template...`，直到全部完成。
- 如果你看到文件数量少于 planned runs，通常是文件名覆盖或失败中断（当前版本已用纳秒时间戳避免同秒覆盖）。

In [37]:
implement_script = FEATURE_IMPLEMENTATION_SCRIPTS / "implement_idea.py"

for idx, (template, idea_text) in enumerate(runs, start=1):
    print(f"[{idx}/{len(runs)}] Implementing template...")
    run_script(
        [
            sys.executable,
            str(implement_script),
            "--template",
            template,
            "--dataset",
            dataset_folder,
            "--idea",
            idea_text,
        ],
        cwd=FEATURE_IMPLEMENTATION_SCRIPTS,
    )

print("Done implementing templates.")


[1/16] Implementing template...
[2/16] Implementing template...
[3/16] Implementing template...
[4/16] Implementing template...
[5/16] Implementing template...
[6/16] Implementing template...
[7/16] Implementing template...
[8/16] Implementing template...
[9/16] Implementing template...
[10/16] Implementing template...
[11/16] Implementing template...
[12/16] Implementing template...
[13/16] Implementing template...
[14/16] Implementing template...
[15/16] Implementing template...
[16/16] Implementing template...
Done implementing templates.


## Step 7) 合并所有 expression 列表
**目的**：把 Step 6 生成的多个 `idea_*.json` 里的表达式统一汇总、去重，生成最终表达式池。

**输入**：
- dataset 目录下所有 `idea_*.json` 文件。

**产出物（落地文件）**：
- `skills/brain-feature-implementation/data/<dataset>_<region>_delay<delay>/final_expressions.json`

**检查点**：
- 代码会打印 `Unique expressions=` 以及前 10 条预览。
- 课堂上可以强调：这个文件是“最终可交付”的表达式候选集合。

In [38]:
merge_script = FEATURE_IMPLEMENTATION_SCRIPTS / "merge_expression_list.py"
run_script(
    [sys.executable, str(merge_script), "--dataset", dataset_folder],
    cwd=FEATURE_IMPLEMENTATION_SCRIPTS,
)

final_path = FEATURE_IMPLEMENTATION_DIR / "data" / dataset_folder / "final_expressions.json"
expressions = json.loads(final_path.read_text(encoding="utf-8"))
print("final_expressions.json path=", final_path)
print("Unique expressions=", len(expressions))
print("Preview (first 10):")
for i, ex in enumerate(expressions[:10], start=1):
    print(f"{i:02d}. {ex}")

final_expressions.json path= D:\BRAINProject\cnhkmcp\cnhkmcp\untracked\APP\trailSomeAlphas\skills\brain-feature-implementation\data\analyst14_GLB_delay1\final_expressions.json
Unique expressions= 246
Preview (first 10):
01. divide(subtract(anl14_high_eps_fp1, anl14_high_eps_fp1), anl14_high_eps_fp1)
02. divide(subtract(anl14_low_eps_fp1, anl14_low_eps_fp1), anl14_low_eps_fp1)
03. divide(subtract(anl14_mean_eps_fp1, anl14_mean_eps_fp1), anl14_mean_eps_fp1)
04. divide(subtract(anl14_median_eps_fp1, anl14_median_eps_fp1), anl14_median_eps_fp1)
05. divide(subtract(anl14_numofests_eps_fp1, anl14_numofests_eps_fp1), anl14_numofests_eps_fp1)
06. divide(subtract(anl14_stddev_eps_fp1, anl14_stddev_eps_fp1), anl14_stddev_eps_fp1)
07. divide(anl14_high_ebit_fy2, anl14_high_ebit_fp1)
08. divide(anl14_high_ebit_fy2, anl14_low_ebit_fp1)
09. divide(anl14_high_ebit_fy2, anl14_mean_ebit_fp1)
10. divide(anl14_high_ebit_fy2, anl14_median_ebit_fp1)


## Step 8) 利用ACE LIB生成表达式
-**目的**：生成符合比赛要求的json

-**注意**：该json并不符合AI比赛要求，需进一步修改

In [39]:
expressions

['divide(subtract(anl14_high_eps_fp1, anl14_high_eps_fp1), anl14_high_eps_fp1)',
 'divide(subtract(anl14_low_eps_fp1, anl14_low_eps_fp1), anl14_low_eps_fp1)',
 'divide(subtract(anl14_mean_eps_fp1, anl14_mean_eps_fp1), anl14_mean_eps_fp1)',
 'divide(subtract(anl14_median_eps_fp1, anl14_median_eps_fp1), anl14_median_eps_fp1)',
 'divide(subtract(anl14_numofests_eps_fp1, anl14_numofests_eps_fp1), anl14_numofests_eps_fp1)',
 'divide(subtract(anl14_stddev_eps_fp1, anl14_stddev_eps_fp1), anl14_stddev_eps_fp1)',
 'divide(anl14_high_ebit_fy2, anl14_high_ebit_fp1)',
 'divide(anl14_high_ebit_fy2, anl14_low_ebit_fp1)',
 'divide(anl14_high_ebit_fy2, anl14_mean_ebit_fp1)',
 'divide(anl14_high_ebit_fy2, anl14_median_ebit_fp1)',
 'divide(anl14_high_ebit_fy2, anl14_numofests_ebit_fp1)',
 'divide(anl14_high_ebit_fy2, anl14_stddev_ebit_fp1)',
 'divide(anl14_low_ebit_fy2, anl14_low_ebit_fp1)',
 'divide(anl14_low_ebit_fy2, anl14_high_ebit_fp1)',
 'divide(anl14_low_ebit_fy2, anl14_mean_ebit_fp1)',
 'divide(

In [40]:
alpha_list = []
for expression in expressions:
    alpha = ace_lib.generate_alpha(
            regular=expression,
            alpha_type="REGULAR",
            region="GLB",
            universe="TOP3000",
            delay=1,
            neutralization="SECTOR",
            decay=5,
            truncation=0.01,
            pasteurization="ON",
            test_period="P0Y0",
            unit_handling="VERIFY",
            nan_handling="ON",
            max_trade="OFF",
            visualization=True,
        )
    alpha_list.append(alpha)

In [41]:
alpha_list

[{'type': 'REGULAR',
  'settings': {'instrumentType': 'EQUITY',
   'region': 'GLB',
   'universe': 'TOP3000',
   'delay': 1,
   'decay': 5,
   'neutralization': 'SECTOR',
   'truncation': 0.01,
   'pasteurization': 'ON',
   'testPeriod': 'P0Y0',
   'unitHandling': 'VERIFY',
   'nanHandling': 'ON',
   'maxTrade': 'OFF',
   'language': 'FASTEXPR',
   'visualization': True},
  'regular': 'divide(subtract(anl14_high_eps_fp1, anl14_high_eps_fp1), anl14_high_eps_fp1)'},
 {'type': 'REGULAR',
  'settings': {'instrumentType': 'EQUITY',
   'region': 'GLB',
   'universe': 'TOP3000',
   'delay': 1,
   'decay': 5,
   'neutralization': 'SECTOR',
   'truncation': 0.01,
   'pasteurization': 'ON',
   'testPeriod': 'P0Y0',
   'unitHandling': 'VERIFY',
   'nanHandling': 'ON',
   'maxTrade': 'OFF',
   'language': 'FASTEXPR',
   'visualization': True},
  'regular': 'divide(subtract(anl14_low_eps_fp1, anl14_low_eps_fp1), anl14_low_eps_fp1)'},
 {'type': 'REGULAR',
  'settings': {'instrumentType': 'EQUITY',
 

## 常见问题排查（按“症状 → 原因 → 怎么看”）
- 症状：Moonshot 明明在输出，但 `ideas_text` 为空
  - 常见原因：key 没生效 / 网络拦截 / base url 配错 / 代理导致 SSE 被截断
  - 怎么看：先确认环境变量 `MOONSHOT_API_KEY`，再看调用返回的错误信息（HTTP 状态码/响应体）
- 症状：`ace_lib` 导入失败或 session 启动失败
  - 常见原因：路径没指到 `scripts` 目录 / 依赖缺失 / 账号配置不对
  - 怎么看：打印 `FEATURE_IMPLEMENTATION_SCRIPTS` 是否指向 `.../skills/brain-feature-implementation/scripts`
- 症状：Step 5 planned runs 数量与 Concept 数量不一致
  - 常见原因：开启了去重（`DEDUP_TEMPLATES=True`）导致重复模板合并
  - 怎么做：想忠实按 Concept 逐条跑，把 `DEDUP_TEMPLATES=False`
- 症状：Step 6 生成的 `idea_*.json` 文件数量不对
  - 常见原因：中途失败停止 / 输出被覆盖 / 输出目录不是你以为的 dataset_folder
  - 怎么看：确认 `dataset_folder`，并检查 stderr/stdout 里是否有失败的那一条模板

流程图代码：以下复制粘贴到https://mermaid-live.nodejs.cn/edit 生成流程图

In [42]:
flowchart TD
  A([开始]) --> P["Step 0 参数设置<br/>DATA_CATEGORY / DATASET_ID / REGION / DELAY / UNIVERSE<br/>DEDUP_TEMPLATES / IDEAS_FILE / REGEN_IDEAS<br/>（可选）MOONSHOT_API_KEY / MOONSHOT_MODEL"]

  P --> S["Step 1 环境与路径准备<br/>导入依赖<br/>定位 skills/scripts 目录<br/>import ace_lib"]
  S --> H["Step 1 辅助函数就绪<br/>读取配置/文本<br/>运行脚本 run_script<br/>抽取 Concept blocks<br/>占位符规范化/校验"]

  H --> B["Step 2 启动 BRAIN 会话<br/>读取 config.json<br/>ace_lib.start_session()<br/>产出：session（内存）"]

  B --> I{"Step 3 ideas 是否可复用？<br/>IDEAS_FILE 存在 且 REGEN_IDEAS=False"}
  I -- 是 --> I1["读取 IDEAS_FILE<br/>产出：ideas_text（内存）"]
  I -- 否 --> I2{"是否提供 MOONSHOT_API_KEY？"}
  I2 -- 否 --> E1["停止：缺少 MOONSHOT_API_KEY<br/>建议：准备好 ideas.md 再继续"]
  I2 -- 是 --> I3["拉取 datasets/datafields/operators<br/>构建 prompt"]
  I3 --> I4["调用 Moonshot/Kimi 生成 ideas<br/>写入 IDEAS_FILE（文件落地）"]
  I4 --> I5["补齐 metadata<br/>Dataset/Region/Delay"]

  I1 --> D0["从 ideas_text 解析 dataset_id_from_file"]
  I5 --> D0

  D0 --> D1["Step 4 下载 dataset CSV<br/>运行 fetch_dataset.py<br/>产出：data/<dataset>_<region>_delay<delay>/*.csv"]
  D1 --> D2["加载 field ids（从 CSV）<br/>推导 suffix candidates<br/>detect dataset_code"]

  D2 --> T1["Step 5 抽取任务列表<br/>Concept blocks → (template, idea)"]
  T1 --> T2["占位符规范化/校验<br/>normalize_template_placeholders<br/>产出：normalized_pairs"]
  T2 --> T3{"是否去重模板？<br/>DEDUP_TEMPLATES"}
  T3 -- True --> T4["按 template 去重<br/>产出：runs（更少、更快）"]
  T3 -- False --> T5["保留每个 Concept 的顺序<br/>产出：runs（更忠实）"]

  T4 --> R["检查点：Planned runs / duplicates"]
  T5 --> R

  R --> IMP["Step 6 执行实现<br/>循环运行 implement_idea.py<br/>--template --dataset --idea<br/>产出：idea_<timestamp>.json（多个文件）"]
  IMP --> M["Step 7 合并结果<br/>运行 merge_expression_list.py<br/>产出：final_expressions.json（文件落地）"]
  M --> V["检查点：Unique expressions 数量<br/>预览前 10 条"]
  V --> Z([结束])

  IMP -->|实现失败| TR1["排查：模板占位符不匹配 / 字段缺失 / operators 问题<br/>看 stderr/stdout + 检查 dataset_folder"]
  M -->|合并失败| TR2["排查：idea_*.json 是否存在/有效<br/>路径是否正确/权限问题"]

SyntaxError: invalid syntax (1353578926.py, line 1)